# 001 - FIFA WORLD CUP

In [2]:
import pandas as pd
import numpy as np
import polars as pl

In [4]:
df_matches = pd.read_csv('Data/WorldCupMatches.csv')
df_players = pd.read_csv('Data/WorldCupPlayers.csv')
df_cups = pd.read_csv('Data/WorldCups.csv')

pl_matches = pl.read_csv('Data/WorldCupMatches.csv')
pl_players = pl.read_csv('Data/WorldCupPlayers.csv')
pl_cups = pl.read_csv('Data/WorldCups.csv')

## 🟢 Desafío 1: El Calentamiento  
### Selección y Filtros

**Objetivo:**  
Familiarizarte con la tabla de Copas del Mundo y practicar selección, filtrado y limpieza básica de datos.

---

### 📌 Ejercicios

1. **Selección básica**  
   Selecciona las siguientes columnas desde la tabla `worldcups`:
   - Año (`Year`)
   - País anfitrión (`Country`)
   - Ganador (`Winner`)

2. **Filtrado por rango de años**  
   Filtra los mundiales jugados entre **1970 y 2000** inclusive.

3. **Reto de limpieza de datos**  
   Algunos nombres de países pueden contener espacios adicionales o caracteres no deseados.  
   Selecciona el nombre del **ganador** asegurándote de que:
   - Esté en **mayúsculas**
   - No tenga espacios en blanco al inicio ni al final

---

### 🧠 Conceptos Clave
- `SELECT`
- `WHERE`
- Funciones de texto (`TRIM`, `UPPER`)

```SQL
SELECT
    UPPER(TRIM(REPLACE(winner,'Germany FR','Germany'))) AS country_winner,
    country AS host_country,
    year
FROM worldcup.cups
WHERE year BETWEEN 1790 and 2000;
```

In [28]:
df_c = df_cups.copy()

df_years = df_c[(df_c['Year'].between(1970,2000))].copy()

df_years['Winner'] = df_years['Winner'].replace('Germany FR','Germany')

df_years['Winner'] = df_years['Winner'].str.strip().str.upper()

respuesta = df_years[['Winner','Country','Year']].reset_index(drop=True)

In [29]:
pl_c = pl_cups

pl_years = pl_c.filter(
    pl.col('Year').is_between(1790,2000)
).with_columns([
    pl.col('Winner').
    replace('Germany FR','Germany')
    .str.strip_chars()
    .str.to_uppercase()
    .alias('Winner')
]).select(['Winner','Country','Year'])

## 🟡 Nivel 2: Estadísticas y Agregaciones  
### El Salto a los Cálculos

**Objetivo:**  
Explorar la potencia de SQL (y herramientas analíticas) para generar métricas a partir de datos históricos de Copas del Mundo.

---

### 🏆 El Reto: *Efectividad Goleadora*

Analiza la tabla `cups` (o `worldcups`) para identificar los mundiales más emocionantes desde el punto de vista ofensivo.

---

### 📌 Requerimientos del Reporte

Genera un reporte que incluya las siguientes columnas:

- **Year**  
  Año del Mundial.

- **Total_Goles**  
  Total de goles anotados en el torneo.  
  *(Usar la columna `GoalsScored`)*

- **Promedio_Goles**  
  Columna calculada que represente el promedio de goles por partido.  
  *(Cálculo: `GoalsScored / MatchesPlayed`)*

---

### 🔍 Filtros

- Mostrar **solo** los mundiales donde el **Total de Goles sea mayor a 120**.

---

### 📊 Ordenamiento

- Ordenar los resultados **de mayor a menor** según el **Promedio_Goles**.

---

### 🧠 Conceptos Clave
- Agregaciones y columnas calculadas
- Operadores aritméticos
- `WHERE`
- `ORDER BY`


```SQL
SELECT
    year,
    goalsscored AS total_goals,
    ROUND((goalsscored * 1.0 /matchesplayed),2) AS avg_goals
FROM worldcup.cups
WHERE goalsscored > 120
ORDER BY total_goals DESC;
```

In [ ]:
df_c = df_cups.copy()

df_g = df_c[(df_c['GoalsScored'] > 120)].copy()

df_g['avg_goals'] = (df_g['GoalsScored'] / df_g['MatchesPlayed']).round(2)

df_g = df_g.sort_values('avg_goals', ascending=False)

respuesta = df_g[['Year','GoalsScored','avg_goals','MatchesPlayed']].reset_index(drop=True)

,Year,GoalsScored,avg_goals,MatchesPlayed
0,1954,140,5.38,26
1,1958,126,3.60,35
2,1982,146,2.81,52
3,1994,141,2.71,52
4,1998,171,2.67,64
5,2014,171,2.67,64
6,1986,132,2.54,52
7,2002,161,2.52,64
8,2006,147,2.30,64
9,2010,145,2.27,64


In [49]:
pl_c = pl_cups

pl_g = pl_c.filter(
    (pl.col('GoalsScored') > 120)
).with_columns(
    (pl.col('GoalsScored') / pl.col('MatchesPlayed')).round(2).alias('avg_goals')
).sort('avg_goals', descending=True).select([
    'Year','GoalsScored','avg_goals','MatchesPlayed'
])

## 🔴 Nivel 3: El Desafío de los JOINs  
### Cruzando Tablas en SQL

**Objetivo:**  
Aprender a unir múltiples tablas en PostgreSQL para construir reportes más ricos combinando información relacionada.

---

### 🔗 El Reto: *La Gran Final*

Necesitamos generar un reporte que combine información de torneos y partidos, enfocándonos **exclusivamente en las Finales**.

---

### 📌 Requerimientos del Reporte

El reporte debe incluir las siguientes columnas:

- **Year**  
  Año del Mundial (desde la tabla `cups`).

- **Winner**  
  País ganador del Mundial (desde la tabla `cups`).

- **Stadium**  
  Estadio donde se jugó la final (desde la tabla `"WorldCupMatches"`).

- **Attendance**  
  Asistencia registrada en la final  
  *(La columna suele llamarse `Attendance` en el dataset de Kaggle)*.

---

### 🔍 Condiciones

- Incluir **solo** los partidos correspondientes a la **Final** del torneo.

---

### 🧠 Reglas Técnicas del JOIN

Al escribir la consulta en **PostgreSQL / DataGrip**, recuerda:

- Utilizar **alias** para las tablas  
  *(Ejemplo: `FROM cups c JOIN "WorldCupMatches" m`)*

- Usar **comillas dobles (`"`)** para nombres de tablas o columnas que contengan:
  - Mayúsculas
  - Espacios

- El **campo común** entre las tablas es:  
  - `year`

---

### 🧠 Conceptos Clave
- `JOIN`
- Alias de tablas
- Filtrado posterior al JOIN
- Manejo de identificadores con comillas dobles


```SQL
SELECT
    c.year,
    c.winner,
    m.stadium,
    m.attendance
FROM worldcup.cups AS c
INNER JOIN worldcup.matches AS m ON c.year = m.year
WHERE LOWER(TRIM(m.stage)) = 'final'
ORDER BY c.year DESC;
```

In [58]:
df_c = df_cups.copy()
df_m = df_matches.copy()

df_merge = df_c.merge(
    df_m, 
    on = 'Year',
     how = 'inner',
     suffixes=('_total','_match')
     )

df_r = df_merge[
    (df_merge['Stage'].str.strip().str.lower() == 'final')
].copy()

df_r = df_r.sort_values(by='Year', ascending=False)

respuesta = df_r[['Year','Winner','Stadium','Attendance_match']]

respuesta

,Year,Winner,Stadium,Attendance_match
828,2014,Germany,Estadio do Maracana,74738.0
851,2014,Germany,Estadio do Maracana,74738.0
771,2010,Spain,Soccer City Stadium,84490.0
707,2006,Italy,Olympiastadion,69000.0
643,2002,Brazil,International Stadium Yokohama,69029.0
579,1998,France,Stade de France,80000.0
515,1994,Brazil,Rose Bowl,94194.0
463,1990,Germany FR,Stadio Olimpico,73603.0
411,1986,Argentina,Estadio Azteca,114600.0
359,1982,Italy,Santiago Bernabeu,90000.0


In [59]:
pl_join = pl_cups.join(
    pl_matches,
    on = 'Year',
    how = 'inner'
).filter(
    pl.col('Stage').str.strip_chars().str.to_lowercase() == 'final'
).sort(
    'Year', descending=True
).select([
    'Year',
    'Winner',
    'Stadium',
    pl.col('Attendance_right').alias('Attendance_match')
])

pl_join

Year,Winner,Stadium,Attendance_match
i64,str,str,i64
2014,"""Germany""","""Estadio do Maracana""",74738
2014,"""Germany""","""Estadio do Maracana""",74738
2010,"""Spain""","""Soccer City Stadium""",84490
2006,"""Italy""","""Olympiastadion""",69000
2002,"""Brazil""","""International Stadium Yokohama""",69029
…,…,…,…
1958,"""Brazil""","""Rasunda Stadium""",49737
1954,"""Germany FR""","""Wankdorf Stadium""",62500
1938,"""Italy""","""Stade Olympique""",45000
